In [ ]:
from utils.imports import *
from datasets import *
from model import *
from training import train
from data_processing import *
from visualization import *
from optimization import *
from others import *

找適合的超參數

In [ ]:
def try_optimization():
    best_params = run_optimization(n_trials=8000,seed=8)
    print("Best parameters:", best_params)

if __name__ == "__main__":
    try_optimization()

訓練模型

In [ ]:
def main():# 可改  
    test_size=0.3
    input_dim = 52 # input_dim
    num_classes = 3
    # seed = 8
    # 超參數
    d_model = 256
    channel_nhead = 2
    nhead = 5
    num_layers = 9
    batch_size = 8
    learning_rate = 1.9764938393830796e-05
    weight_decay = 0.0003118927619072748
    dropout = 0.6571041748645935
    embedding_dropout = 0.24177241045928943
    dim_feedforward = 640
    patience = 83
    grad_clip_value = 3.166491767088093

    num_epochs = 1000    
    use_augmentation = False
    # 確保 d_model 能被 nhead 跟 channel_nhead 整除
    d_model = adjust_d_model(d_model, channel_nhead, nhead)

    # set_seed(seed)
    # 創建固定的生成器
    # g = torch.Generator()
    # g.manual_seed(seed)
    # 設置設備
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # 加載數據
    dataset = OxyDataset(root_dir=r'E:\JoFz\GTC4000\data')
    
    # 處理類別不平衡
    class_counts = np.bincount(dataset.targets)
    class_weights = 1. / class_counts
    sample_weights = class_weights[dataset.targets]

    # 分割數據
    # train_indices, test_indices = train_test_split(range(len(dataset)), test_size=test_size, stratify=dataset.targets, random_state=seed)
    train_indices, test_indices = train_test_split(range(len(dataset)), test_size=test_size, stratify=dataset.targets)
    train_dataset = Subset(dataset, train_indices)
    test_dataset = Subset(dataset, test_indices)

    # train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=WeightedRandomSampler(sample_weights[train_indices], len(train_indices), generator=g), generator=g)
    # test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, generator=g)
    train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=WeightedRandomSampler(sample_weights[train_indices], len(train_indices)))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    # 創建模型
    # set_seed(seed)
    # model = TransformerModel(input_dim, d_model, nhead, num_layers, num_classes, dropout, embedding_dropout, dim_feedforward, seed).to(device)
    model = TransformerModel(input_dim, d_model, nhead, num_layers, num_classes, channel_nhead, dropout, embedding_dropout, dim_feedforward).to(device)

    # 定義loss function和optimizer
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float).to(device))
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    T_max = num_epochs  # 設置總訓練epoch數
    eta_min = 1e-6  # 最小學習率
    scheduler = CosineAnnealingLR(optimizer, T_max=T_max, eta_min=eta_min)

    model, train_losses, val_losses = train(model, train_loader, test_loader, criterion, optimizer, 
                                            scheduler, num_epochs, device, use_augmentation, 
                                            grad_clip_value, patience)
    
    # 評估模型
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}')

    plot_confusion_matrix(model, test_loader, device)
    plot_loss_curves(train_losses, val_losses)

if __name__ == '__main__':
    main()